# Basketball Data Discovery — Game 114086

This notebook is a first tour of one SkillCorner game. It shows how to turn the game metadata, Dynamic Events JSON, and compressed tracking JSON Lines into pandas DataFrames. The goal is to get comfortable with the structure before building a shot-quality model.

**Game:** Dreamland Gran Canaria vs. Bitci Baskonia, 2026-02-08. Coordinates and distances are in feet; tracking is sampled at 25 frames per second.

## 1. Imports and find the data
The path setup works if Jupyter starts in the repository root or in the `notebooks/` folder. Change `GAME_ID` to explore another game folder later.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

GAME_ID = '114086'
candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in candidates if (p / 'data' / 'matches' / GAME_ID).is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Could not locate data/matches/' + GAME_ID + ' from ' + str(Path.cwd()))

GAME_DIR = REPO_ROOT / 'data' / 'matches' / GAME_ID
GAME_DATA_PATH = GAME_DIR / f'{GAME_ID}_game_data.json'
EVENTS_PATH = GAME_DIR / f'{GAME_ID}_dynamic_events.json'
TRACKING_PATH = GAME_DIR / f'{GAME_ID}_tracking_data.jsonl.gz'
print('Repository:', REPO_ROOT)
print('Game files:')
for path in (GAME_DATA_PATH, EVENTS_PATH, TRACKING_PATH):
    print(f'  {path.name}: {path.exists()}')

Repository: /Users/sameerprasadkoppolu/Desktop/Sports Writing/sports_writing_nba/sports analytics cup 2/opendata-basketball-skillcorner
Game files:
  114086_game_data.json: True
  114086_dynamic_events.json: True
  114086_tracking_data.jsonl.gz: True


## 2. Game metadata and rosters
The game-data file is a single JSON object, not a table of events. We can make a one-row game summary and separate home/away roster DataFrames. `playerId` is the join key used by the event tables.

game_data --> Contains a JSON that has GameID, Datem Competition, Season, and the Rosters of Home and Away

In [2]:
with GAME_DATA_PATH.open(encoding='utf-8') as f:
    game_data = json.load(f)

game_data

{'gameId': 114086,
 'date': '2026-02-08T17:00:00Z',
 'competitionEditionId': 211,
 'competitionId': 8,
 'competitionName': 'Liga ACB',
 'seasonId': 51,
 'seasonName': '2025-2026',
 'dynamicEventsLastProcessingDate': '2026-07-17T12:23:44Z',
 'trackingLastProcessingDate': '2026-07-17T12:23:16Z',
 'physicalLastProcessingDate': '2026-07-17T12:23:06Z',
 'homeTeam': {'teamId': 1328,
  'teamName': 'Dreamland Gran Canaria',
  'players': [{'playerId': 15347,
    'firstName': 'Isaiah',
    'lastName': 'Wong',
    'jersey': '2'},
   {'playerId': 19426,
    'firstName': 'Kur',
    'lastName': 'Kuath',
    'jersey': '23'},
   {'playerId': 25182, 'firstName': 'Eric', 'lastName': 'Vila', 'jersey': '4'},
   {'playerId': 32336,
    'firstName': 'Kassius',
    'lastName': 'Robertson',
    'jersey': '30'},
   {'playerId': 35402,
    'firstName': 'Nico',
    'lastName': 'Brussino',
    'jersey': '9'},
   {'playerId': 59153,
    'firstName': 'Louis',
    'lastName': 'Labeyrie',
    'jersey': '21'},
   {'pl

In [3]:
game_summary = pd.DataFrame([{
    key: game_data.get(key)
    for key in ('gameId', 'date', 'competitionName', 'seasonName', 'competitionId', 'seasonId')
}])
display(game_summary)

def roster_frame(team_key):
    team = game_data[team_key]
    roster = pd.DataFrame(team.get('players', []))
    roster['teamId'] = team.get('teamId')
    roster['teamName'] = team.get('teamName')
    return roster

home_roster = roster_frame('homeTeam')
away_roster = roster_frame('awayTeam')
roster = pd.concat([home_roster, away_roster], ignore_index=True)
print('Home roster:', game_data['homeTeam']['teamName'], home_roster.shape)
display(home_roster.head())
print('Away roster:', game_data['awayTeam']['teamName'], away_roster.shape)
display(away_roster.head())

,gameId,date,competitionName,seasonName,competitionId,seasonId
0,114086,2026-02-08T17:00:00Z,Liga ACB,2025-2026,8,51


Home roster: Dreamland Gran Canaria (12, 6)


,playerId,firstName,lastName,jersey,teamId,teamName
0,15347,Isaiah,Wong,2,1328,Dreamland Gran Canaria
1,19426,Kur,Kuath,23,1328,Dreamland Gran Canaria
2,25182,Eric,Vila,4,1328,Dreamland Gran Canaria
3,32336,Kassius,Robertson,30,1328,Dreamland Gran Canaria
4,35402,Nico,Brussino,9,1328,Dreamland Gran Canaria


Away roster: Bitci Baskonia (11, 6)


,playerId,firstName,lastName,jersey,teamId,teamName
0,22724,Eugene,Omoruyi,5,1326,Bitci Baskonia
1,24268,Trent,Forrest,11,1326,Bitci Baskonia
2,24597,Mamadi,Diakite,1,1326,Bitci Baskonia
3,24949,Markus,Howard,0,1326,Bitci Baskonia
4,34992,Timothe,Luwawu-Cabarrot,9,1326,Bitci Baskonia


## 3. Dynamic Events: many related tables in one JSON
Dynamic Events is a dictionary whose keys are event families (`shots`, `passes`, `touches`, `chances`, and others). Each value is a list of records. Convert each list to its own DataFrame: they have different grains and should not be stacked into one giant table. Empty event families become empty DataFrames.

In [4]:
with EVENTS_PATH.open(encoding='utf-8') as f:
    events_json = json.load(f)
events_json

{'chances': [{'id': 'chance-114086-1-0',
   'possessionId': 'possession-114086-1-0',
   'gameId': 114086,
   'season': '2025-2026',
   'period': 1,
   'startFrame': 501,
   'endFrame': 820,
   'startWallClock': 20040,
   'endWallClock': 32800,
   'startGameClock': 600.0,
   'endGameClock': 587.26171875,
   'offTeamId': 1326,
   'defTeamId': 1328,
   'startType': 'JMP',
   'outcome': 'FOU',
   'offPlayerIds': [24949, 24268, 59136, 59138, 35096],
   'defPlayerIds': [59157, 15347, 25182, 59158, 35402],
   'homeStartScore': 0,
   'awayStartScore': 0,
   'ptsScored': 0,
   'startShotClock': None,
   'endShotClock': 11.6,
   'ballInPaint': True,
   'endOfPossession': False,
   'endLoc': [-29.194348753280828, 8.900559793307085],
   'frontcourtFrame': 613,
   'transition': False,
   'lastChanceId': None,
   'qualityIndex': 5,
   'usable': True},
  {'id': 'chance-114086-1-1',
   'possessionId': 'possession-114086-1-0',
   'gameId': 114086,
   'season': '2025-2026',
   'period': 1,
   'startFram

In [5]:
print('Event families:', list(events_json))
event_frames = {name: pd.DataFrame(records) for name, records in events_json.items()}
event_overview = pd.DataFrame([
    {'event_family': name, 'rows': len(frame), 'columns': len(frame.columns)}
    for name, frame in event_frames.items()
]).sort_values('event_family').reset_index(drop=True)
display(event_overview)

# Convenience references for common tables
shots = event_frames['shots']
chances = event_frames['chances']
touches = event_frames['touches']
matchups = event_frames['matchups']
print('Shots shape:', shots.shape)
display(shots.head(3))

Event families: ['chances', 'possessions', 'shots', 'rebounds', 'turnovers', 'free_throws', 'fouls', 'timeouts', 'passes', 'touches', 'dribbles', 'picks', 'isolations', 'handoffs', 'drives', 'posts', 'off_ball_screens', 'closeouts', 'chance_players', 'matchups']


,event_family,rows,columns
0,chance_players,1950,14
1,chances,195,30
2,closeouts,66,31
3,dribbles,1336,15
4,drives,84,28
5,fouls,37,24
6,free_throws,32,15
7,handoffs,49,25
8,isolations,19,23
9,matchups,1347,15


Shots shape: (139, 40)


,id,gameId,season,period,chanceId,possessionId,startFrame,startWallClock,startGameClock,endFrame,...,complexShotType,catchAndShoot,shotQuality,createdFromPaint,contested,contestLevel,releaseTime,closestDefDist,closestDefId,touchId
0,shot-114086-1-0,114086,2025-2026,1,chance-114086-1-1,possession-114086-1-0,1380,55200,580.262756,1395,...,layup,False,48.741230,True,True,plus,2.12,0.554495,59158.0,touch-114086-1-6
1,shot-114086-1-1,114086,2025-2026,1,chance-114086-1-2,possession-114086-1-0,1437,57480,577.983032,1455,...,hook,None,47.574078,True,True,plus,NaN,2.611572,59157.0,NaN
2,shot-114086-1-2,114086,2025-2026,1,chance-114086-1-3,possession-114086-1-1,1942,77680,557.785339,1970,...,catchAndShoot,True,35.904671,False,True,plus,0.68,4.946294,59138.0,touch-114086-1-11


### Inspect columns and data types
This helps identify which fields are numeric, categorical, booleans, or nested values such as `[x, y]` locations. Different event families have different columns.

In [6]:
for name in ('shots', 'touches', 'matchups', 'picks', 'drives'):
    frame = event_frames.get(name, pd.DataFrame())
    print(f'\n{name}: {frame.shape}')
    print(frame.dtypes.to_string())

print('Missing values in shots (largest counts):')
display(shots.isna().sum().sort_values(ascending=False).head(12).rename('missing').to_frame())


shots: (139, 40)
id                      str
gameId                int64
season                  str
period                int64
chanceId                str
possessionId            str
startFrame            int64
startWallClock        int64
startGameClock      float64
endFrame              int64
endWallClock          int64
endGameClock        float64
location             object
region                  str
shotClock           float64
shooterId             int64
offTeamId             int64
defTeamId             int64
three                  bool
outcome                bool
fouled                 bool
foulId                  str
assisted               bool
assistOpp              bool
passerId            float64
blocked                bool
blockerId           float64
distance            float64
dribblesBefore      float64
shotType                str
complexShotType         str
catchAndShoot        object
shotQuality         float64
createdFromPaint       bool
contested            object
co

,missing
blockerId,137
foulId,123
passerId,15
releaseTime,8
catchAndShoot,7
dribblesBefore,7
touchId,7
shotQuality,4
contested,4
complexShotType,4


### Make a more readable shots DataFrame
The event table stores player IDs. Joining the roster adds names. This is a **many-to-one** join: each shot has one shooter, and that player ID should match one roster row. We validate the join to surface unexpected ID issues.

In [7]:
shooter_names = roster[['playerId', 'firstName', 'lastName', 'teamName']].drop_duplicates('playerId')
shots_named = shots.merge(
    shooter_names, left_on='shooterId', right_on='playerId',
    how='left', validate='many_to_one', suffixes=('', '_roster')
)
shots_named['shooterName'] = (shots_named['firstName'].fillna('') + ' ' + shots_named['lastName'].fillna('')).str.strip()
display_cols = [c for c in ['shooterName', 'teamName', 'period', 'startGameClock', 'region', 'distance', 'closestDefDist', 'contestLevel', 'shotType', 'complexShotType', 'three', 'outcome', 'shotQuality'] if c in shots_named.columns]
display(shots_named[display_cols].head(10))
print('Shooter IDs without roster match:', shots_named['shooterName'].eq('').sum())

,shooterName,teamName,period,startGameClock,region,distance,closestDefDist,contestLevel,shotType,complexShotType,three,outcome,shotQuality
0,Rodions Kurucs,Bitci Baskonia,1,580.262756,ra,2.367698,0.554495,plus,layup,layup,False,False,48.741230
1,Rodions Kurucs,Bitci Baskonia,1,577.983032,key,8.471460,2.611572,plus,post,hook,False,True,47.574078
2,Mike Tobey,Dreamland Gran Canaria,1,557.785339,middle two,17.317250,4.946294,plus,jumper,catchAndShoot,False,False,35.904671
3,Khalifa Diop,Bitci Baskonia,1,542.666992,key,4.778515,1.412428,plus,post,hook,False,False,28.711561
4,Isaiah Wong,Dreamland Gran Canaria,1,529.188538,key,5.302887,1.438586,average,jumper,dribblePullUp,False,False,30.245832
5,Khalifa Diop,Bitci Baskonia,1,517.589844,ra,1.761310,3.400991,average,layup,layup,False,False,30.044470
6,Ziga Samar,Dreamland Gran Canaria,1,502.109680,left corner three,22.005117,5.921801,average,jumper,catchAndShoot,True,False,40.266529
7,Markus Howard,Bitci Baskonia,1,495.189514,ra,3.583502,1.514338,plus,layup,layup,False,True,66.312096
8,Mike Tobey,Dreamland Gran Canaria,1,474.309021,ra,3.148293,2.087725,average,layup,tip,False,False,34.514973
9,Rodions Kurucs,Bitci Baskonia,1,462.240387,right wing three,24.822205,5.893439,average,jumper,catchAndShoot,True,False,36.289528


Shooter IDs without roster match: 0


### Join shots to chances
`chanceId` connects each shot to its containing scoring opportunity. A chance can include more than one event (for example, an offensive rebound creates a new chance within the same possession). Select chance context columns first to avoid confusing duplicate columns after the merge.

In [8]:
chance_columns = [c for c in ['id', 'startType', 'outcome', 'transition', 'qualityIndex', 'usable', 'startShotClock', 'endShotClock', 'ptsScored'] if c in chances.columns]
chance_context = chances[chance_columns].rename(columns={'id': 'chanceId'})
shots_with_chance = shots_named.merge(
    chance_context, on='chanceId', how='left', validate='many_to_one', suffixes=('', '_chance')
)
print('Joined rows:', len(shots_with_chance), '| Original shots:', len(shots))
display(shots_with_chance[[c for c in ['startType', 'shooterName', 'region', 'distance', 'contestLevel', 'transition', 'startShotClock', 'outcome', 'shotQuality'] if c in shots_with_chance.columns]].head(10))

Joined rows: 139 | Original shots: 139


,startType,shooterName,region,distance,contestLevel,transition,startShotClock,outcome,shotQuality
0,DEFOB,Rodions Kurucs,ra,2.367698,plus,False,14.00,False,48.741230
1,FGORB,Rodions Kurucs,key,8.471460,plus,False,14.00,True,47.574078
2,FGM,Mike Tobey,middle two,17.317250,plus,False,12.16,False,35.904671
3,FGDRB,Khalifa Diop,key,4.778515,plus,False,24.00,False,28.711561
4,FGDRB,Isaiah Wong,key,5.302887,average,True,24.00,False,30.245832
5,FGDRB,Khalifa Diop,ra,1.761310,average,False,24.00,False,30.044470
6,FTM,Ziga Samar,left corner three,22.005117,average,False,23.84,False,40.266529
7,FGDRB,Markus Howard,ra,3.583502,plus,True,24.00,True,66.312096
8,FGM,Mike Tobey,ra,3.148293,average,False,24.00,False,34.514973
9,DEFOB,Rodions Kurucs,right wing three,24.822205,average,False,24.00,False,36.289528


In [14]:
list(events_json.keys())

['chances',
 'possessions',
 'shots',
 'rebounds',
 'turnovers',
 'free_throws',
 'fouls',
 'timeouts',
 'passes',
 'touches',
 'dribbles',
 'picks',
 'isolations',
 'handoffs',
 'drives',
 'posts',
 'off_ball_screens',
 'closeouts',
 'chance_players',
 'matchups']

## 4. Tracking data: one DataFrame row per video frame
Tracking is gzip-compressed JSON Lines: each JSON object is one frame. Pandas can read it directly. The nested `homePlayers` and `awayPlayers` lists contain player positions; `ball` is another nested object. Some rows represent dead time and have empty lists/objects, so not every frame has player or ball coordinates. Reading the complete game uses more memory than reading event tables, but avoids manually decompressing the file.

**Git LFS note:** the tracking files are stored with Git LFS. If the next cell says this file is a Git LFS pointer, the actual data has not been fetched to your checkout. Install Git LFS as a system tool (not in a Python environment), then from the repository root run `git lfs install` and `git lfs pull`; rerun the tracking cell afterward.

In [9]:
# Git LFS pointer files are plain text, not the underlying compressed tracking file.
with TRACKING_PATH.open('rb') as f:
    tracking_header = f.read(128)

if tracking_header.startswith(b'version https://git-lfs.github.com/spec/v1'):
    raise RuntimeError(
        f'{TRACKING_PATH.name} is a Git LFS pointer, not the tracking data. '
        'Fetch the repository LFS objects, then rerun this cell. '
        'The file should be a gzip stream rather than a short text file.'
    )

tracking = pd.read_json(TRACKING_PATH, lines=True, compression='gzip')
print('Tracking shape:', tracking.shape)
print('Approx. memory (MB):', round(tracking.memory_usage(deep=True).sum() / 1_000_000, 1))
print('Columns:', tracking.columns.tolist())
display(tracking.head(3))

tracking['home_player_count'] = tracking['homePlayers'].map(len)
tracking['away_player_count'] = tracking['awayPlayers'].map(len)
tracking['has_ball_position'] = tracking['ball'].map(bool)
print('Frames with no player positions:', ((tracking.home_player_count + tracking.away_player_count) == 0).sum())
print('Frames with ball position:', tracking['has_ball_position'].sum(), 'of', len(tracking))
display(tracking[['frameIdx', 'period', 'gameClock', 'shotClock', 'home_player_count', 'away_player_count', 'has_ball_position']].head())

Tracking shape: (124796, 9)
Approx. memory (MB): 52.9
Columns: ['frameIdx', 'wallClock', 'gameClock', 'gameClockStopped', 'period', 'homePlayers', 'awayPlayers', 'ball', 'shotClock']


,frameIdx,wallClock,gameClock,gameClockStopped,period,homePlayers,awayPlayers,ball,shotClock
0,0,0,600.0,True,1,"[{'xyz': [6.007397268700787, 9.643875082020998...","[{'xyz': [-0.545872165149278, -7.6253895997375...","{'xyz': [0.8963036724901571, 12.9759781003937,...",NaN
1,1,40,600.0,True,1,"[{'xyz': [6.290626025262467, 9.928385416666666...","[{'xyz': [-0.7081519900344481, -7.548494914698...","{'xyz': [0.679236384514435, 13.0246780675853, ...",NaN
2,2,80,600.0,True,1,"[{'xyz': [6.573854781824147, 10.21289575131233...","[{'xyz': [-0.8704318149196191, -7.471600229658...","{'xyz': [0.462169096538713, 13.073378034776901...",NaN


Frames with no player positions: 32608
Frames with ball position: 92188 of 124796


,frameIdx,period,gameClock,shotClock,home_player_count,away_player_count,has_ball_position
0,0,1,600.0,NaN,5,5,True
1,1,1,600.0,NaN,5,5,True
2,2,1,600.0,NaN,5,5,True
3,3,1,600.0,NaN,5,5,True
4,4,1,600.0,NaN,5,5,True


### Expand one frame into player and ball DataFrames
This is a useful first look at the nested tracking schema without exploding the entire game into a much larger table. `frameIdx` is the tracking key that lines up with event `startFrame` values; both use the same 25 fps video timeline.

In [15]:
# Pick the first frame that actually has players, rather than a dead-time frame.
live_frame = tracking.loc[(tracking.home_player_count + tracking.away_player_count) > 0].iloc[0]
frame_context = {key: live_frame[key] for key in ['frameIdx', 'wallClock', 'gameClock', 'shotClock', 'period']}

def player_rows(players, team_side, context):
    rows = []
    for player in players:
        x, y, z = player.get('xyz', [None, None, None])
        rows.append({**context, 'teamSide': team_side, 'playerId': player.get('playerId'),
                     'jersey': player.get('jersey'), 'x': x, 'y': y, 'z': z,
                     'speed': player.get('speed'), 'isDetected': player.get('isDetected'),
                     'predError': player.get('predError')})
    return rows

one_frame_players = pd.DataFrame(
    player_rows(live_frame['homePlayers'], 'home', frame_context) +
    player_rows(live_frame['awayPlayers'], 'away', frame_context)
)
ball = live_frame['ball']
if ball and ball.get('xyz') is not None:
    ball_x, ball_y, ball_z = ball['xyz']
    one_frame_ball = pd.DataFrame([{**frame_context, 'x': ball_x, 'y': ball_y, 'z': ball_z,
                                   'speed': ball.get('speed'), 'isDetected': ball.get('isDetected'),
                                   'predError': ball.get('predError')}])
else:
    one_frame_ball = pd.DataFrame(columns=['frameIdx', 'period', 'x', 'y', 'z', 'speed', 'isDetected', 'predError'])

print('Frame:', frame_context, '| player rows:', one_frame_players.shape, '| ball rows:', one_frame_ball.shape)
display(one_frame_players.head())
display(one_frame_ball)

Frame: {'frameIdx': np.int64(0), 'wallClock': np.int64(0), 'gameClock': np.float64(600.0), 'shotClock': np.float64(nan), 'period': np.int64(1)} | player rows: (10, 14) | ball rows: (1, 11)


,frameIdx,wallClock,gameClock,shotClock,period,teamSide,playerId,jersey,x,y,z,speed,isDetected,predError
0,0,0,600.0,NaN,1,home,35402,9,6.007397,9.643875,0.0,5.020582,1,2.585584
1,0,0,600.0,NaN,1,home,25182,4,2.111400,-4.075418,0.0,2.223538,0,1.537894
2,0,0,600.0,NaN,1,home,15347,2,-2.744179,-1.947999,0.0,1.086938,1,0.464973
3,0,0,600.0,NaN,1,home,59157,18,-8.637836,-4.354162,0.0,0.242899,1,0.149684
4,0,0,600.0,NaN,1,home,59158,5,-6.984601,-0.735305,0.0,1.189465,1,0.362446


,frameIdx,wallClock,gameClock,shotClock,period,x,y,z,speed,isDetected,predError
0,0,0,600.0,NaN,1,0.896304,12.975978,4.277267,3.444241,0.0,-0.591528


### Connect a shot event to its tracking frame
Shots use `startFrame` for release; tracking uses `frameIdx`. Try changing `SHOT_ROW` to inspect another attempt. An event row may not have a usable tracking counterpart, so check for a match before relying on it.

In [17]:
shots_named.keys()

Index(['id', 'gameId', 'season', 'period', 'chanceId', 'possessionId',
       'startFrame', 'startWallClock', 'startGameClock', 'endFrame',
       'endWallClock', 'endGameClock', 'location', 'region', 'shotClock',
       'shooterId', 'offTeamId', 'defTeamId', 'three', 'outcome', 'fouled',
       'foulId', 'assisted', 'assistOpp', 'passerId', 'blocked', 'blockerId',
       'distance', 'dribblesBefore', 'shotType', 'complexShotType',
       'catchAndShoot', 'shotQuality', 'createdFromPaint', 'contested',
       'contestLevel', 'releaseTime', 'closestDefDist', 'closestDefId',
       'touchId', 'playerId', 'firstName', 'lastName', 'teamName',
       'shooterName'],
      dtype='str')

In [16]:
SHOT_ROW = 0  # Change this to another row number from shots_named
shot = shots_named.iloc[SHOT_ROW]
release_frame = int(shot['startFrame']) if pd.notna(shot.get('startFrame')) else None
print('Shooter:', shot.get('shooterName'), '| event release frame:', release_frame)

if release_frame is not None:
    tracking_at_release = tracking.loc[tracking['frameIdx'].eq(release_frame)]
    if tracking_at_release.empty:
        print('No tracking row found for this release frame.')
    else:
        print('Matching tracking rows:', len(tracking_at_release))
        display(tracking_at_release[['frameIdx', 'period', 'gameClock', 'homePlayers', 'awayPlayers', 'ball']])
        print('Shot event location:', shot.get('location'), '| tracking frame player locations are nested in homePlayers/awayPlayers')

Shooter: Rodions Kurucs | event release frame: 1380
Matching tracking rows: 1


,frameIdx,period,gameClock,homePlayers,awayPlayers,ball
1380,1380,1,580.263,"[{'xyz': [36.88381725721785, -11.9058604002624...","[{'xyz': [37.088869750656166, -20.082328576115...","{'xyz': [39.5751312335958, 3.594826525590551, ..."


Shot event location: [-40.0877624671916, -2.273199126476378] | tracking frame player locations are nested in homePlayers/awayPlayers


### Optional: flatten player positions for every frame
Only run this when you want a long-form tracking DataFrame for analysis. It creates one row per tracked player per frame and can use substantial memory. The default notebook does not execute the expansion.

In [18]:
def iter_all_player_rows(frame_table):
    for row in frame_table.itertuples(index=False):
        context = {'frameIdx': row.frameIdx, 'period': row.period, 'gameClock': row.gameClock, 'shotClock': row.shotClock}
        yield from player_rows(row.homePlayers, 'home', context)
        yield from player_rows(row.awayPlayers, 'away', context)

# Set to True only when you are ready to build the larger table.
BUILD_FULL_PLAYER_TRACKING = False
if BUILD_FULL_PLAYER_TRACKING:
    all_player_positions = pd.DataFrame(iter_all_player_rows(tracking))
    print('Long-form player tracking shape:', all_player_positions.shape)
    display(all_player_positions.head())
else:
    print('Skipped full-game expansion. Set BUILD_FULL_PLAYER_TRACKING = True to build it.')

Skipped full-game expansion. Set BUILD_FULL_PLAYER_TRACKING = True to build it.


## What you have learned / next steps
- Game metadata is a single object; rosters become small player DataFrames keyed by `playerId`.
- Dynamic Events are multiple related DataFrames, one per event family. `gameId`, `possessionId`, `chanceId`, `touchId`, player IDs, and event frame/time fields connect them.
- Tracking is frame-level, with nested player lists and a ball object. Expand nested data only when needed. Coordinates are in feet, and the timeline is 25 fps. Empty tracking arrays often indicate dead time.
- For a shot-quality project, `shots.outcome` is the made/missed result, while `shots.shotQuality` is SkillCorner's existing 0–100 score. Treat that supplied score as a benchmark or comparison—not as a predictor of made/missed outcomes, since that would leak a precomputed quality estimate into the feature set.

A sensible next notebook can build one row per shot, then add pre-shot tracking features and contextual event features. Keep the shot outcome as the label for a made/missed model, and split evaluation by game/player where practical.